In [ ]:
# ============================================================================
# BLOCK 1: Install GROMACS 2024.1 with CUDA (L4 Optimized)
# ============================================================================
# Runtime: ~25 minutes on L4 (faster than T4)

import subprocess
import sys
import os

print("Installing GROMACS 2024.1 with CUDA for L4...")
print("Expected time: ~25 minutes\n")

# Install dependencies
print("Step 1/4: Installing dependencies...")
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'cmake', 'build-essential', 'libfftw3-dev', 'wget'], check=True)

# Download GROMACS 2024.1
print("\nStep 2/4: Downloading GROMACS 2024.1...")
subprocess.run(['wget', '-q',
                'ftp://ftp.gromacs.org/gromacs/gromacs-2024.1.tar.gz',
                '-O', '/tmp/gromacs-2024.1.tar.gz'], check=True)
subprocess.run(['tar', 'xzf', '/tmp/gromacs-2024.1.tar.gz', '-C', '/tmp/'], check=True)

# Build with CUDA (L4 has newer CUDA, more cores)
print("\nStep 3/4: Building GROMACS with CUDA...")
os.makedirs('/tmp/gromacs-2024.1/build', exist_ok=True)
os.chdir('/tmp/gromacs-2024.1/build')

cmake_result = subprocess.run([
    'cmake', '..',
    '-DGMX_BUILD_OWN_FFTW=ON',
    '-DGMX_GPU=CUDA',
    '-DCUDA_TOOLKIT_ROOT_DIR=/usr/local/cuda',
    '-DCMAKE_INSTALL_PREFIX=/usr/local',
    '-DGMX_MPI=off',
    '-DCMAKE_BUILD_TYPE=Release',
    '-DGMX_CUDA_TARGET_SM=89',  # L4 compute capability
], capture_output=True, text=True)

if cmake_result.returncode != 0:
    print("CMAKE FAILED:")
    print(cmake_result.stderr)
    sys.exit(1)

# L4 has more CPU cores - use them all
cpu_count = os.cpu_count() or 2
print(f"Building with {cpu_count} cores (L4 has more than T4)...")

make_result = subprocess.run(['make', f'-j{cpu_count}'],
                             capture_output=True, text=True)
if make_result.returncode != 0:
    print("BUILD FAILED:")
    print(make_result.stderr[-2000:])
    sys.exit(1)

print("\nStep 4/4: Installing...")
subprocess.run(['make', 'install'], check=True, capture_output=True)

# Set environment
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

print("\n" + "="*70)
print("✅ GROMACS 2024.1 INSTALLED!")
print("="*70)

# Verify
version_result = subprocess.run(['gmx', '-version'], capture_output=True, text=True)
for line in version_result.stdout.split('\n')[:5]:
    print(line)

Installing GROMACS 2024.1 with CUDA for L4...
Expected time: ~25 minutes

Step 1/4: Installing dependencies...

Step 2/4: Downloading GROMACS 2024.1...

Step 3/4: Building GROMACS with CUDA...
Building with 12 cores (L4 has more than T4)...

Step 4/4: Installing...

✅ GROMACS 2024.1 INSTALLED!
                         :-) GROMACS - gmx, 2024.1 (-:

Executable:   /usr/local/bin/gmx
Data prefix:  /usr/local
Working dir:  /tmp/gromacs-2024.1/build


In [ ]:
# ============================================================================
# BLOCK 2: Mount Google Drive and Setup Directories
# ============================================================================

from google.colab import drive
import os

# ═══════════════════════════════════════════════════════════════════════
# CONFIGURATION - CHANGE THIS
# ═══════════════════════════════════════════════════════════════════════
COMPOUND_FOLDER = "96_MD"  # Change to "96_MD" for second compound
# ═══════════════════════════════════════════════════════════════════════

# Mount Drive
if not os.path.ismount('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

# Setup directories
DRIVE_DIR = f"/content/drive/MyDrive/{COMPOUND_FOLDER}"
LOCAL_DIR = f"/content/{COMPOUND_FOLDER}"

os.makedirs(LOCAL_DIR, exist_ok=True)
os.chdir(LOCAL_DIR)

print(f"\n{'='*70}")
print(f"Working Directory: {LOCAL_DIR}")
print(f"Backup Location:   {DRIVE_DIR}")
print(f"{'='*70}")

# Check TPR files
print("\nChecking TPR files in Drive...")
for rep in ["md_rep1", "md_rep2", "md_rep3"]:
    tpr_path = f"{DRIVE_DIR}/{rep}.tpr"
    if os.path.exists(tpr_path):
        size_mb = os.path.getsize(tpr_path) / (1024*1024)
        print(f"  ✓ {rep}.tpr ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ MISSING: {rep}.tpr")

Mounting Google Drive...
Mounted at /content/drive

Working Directory: /content/96_MD
Backup Location:   /content/drive/MyDrive/96_MD

Checking TPR files in Drive...
  ✓ md_rep1.tpr (3.1 MB)
  ✓ md_rep2.tpr (3.1 MB)
  ✓ md_rep3.tpr (3.1 MB)


In [ ]:
# ============================================================================
# BLOCK 3: Verify L4 GPU Detection
# ============================================================================

import subprocess
import sys

print("="*70)
print("L4 GPU DETECTION TEST")
print("="*70)

# Test 1: NVIDIA GPU
print("\nTest 1: Hardware Detection")
nvidia_result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if nvidia_result.returncode != 0:
    print("❌ No GPU detected!")
    print("Check: Runtime → Change runtime type → GPU")
    sys.exit(1)

# Check for L4
lines = nvidia_result.stdout.split('\n')
is_l4 = False
for line in lines:
    if 'L4' in line:
        print(f"  ✅ L4 GPU DETECTED: {line.strip()}")
        is_l4 = True
        break
    elif 'T4' in line:
        print(f"  ⚠️  T4 detected (not L4): {line.strip()}")
        print("     T4 will work but L4 is faster and more reliable")
        break

# Test 2: GROMACS CUDA Support
print("\nTest 2: GROMACS CUDA Support")
gmx_result = subprocess.run(['gmx', 'mdrun', '-version'],
                           capture_output=True, text=True)

if 'CUDA' in gmx_result.stdout:
    print("  ✅ CUDA support enabled")
    for line in gmx_result.stdout.split('\n'):
        if 'GPU support' in line or 'CUDA' in line or 'NBNxM' in line:
            print(f"     {line.strip()}")
else:
    print("  ❌ No CUDA support!")
    sys.exit(1)

# Test 3: Can we use GPU mode?
test_result = subprocess.run(['gmx', 'mdrun', '-nb', 'gpu', '-h'],
                            capture_output=True, text=True, timeout=10)
if test_result.returncode == 0:
    print("\nTest 3: GPU mode READY ✅")
else:
    print("\nTest 3: GPU mode ERROR ❌")
    sys.exit(1)

print("\n" + "="*70)
if is_l4:
    print("🚀 L4 VERIFIED - OPTIMAL PERFORMANCE EXPECTED!")
else:
    print("🚀 GPU VERIFIED - READY TO RUN!")
print("="*70)

L4 GPU DETECTION TEST

Test 1: Hardware Detection
  ✅ L4 GPU DETECTED: |   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |

Test 2: GROMACS CUDA Support
  ✅ CUDA support enabled
     GPU support:         CUDA
     NBNxM GPU setup:     super-cluster 2x2x2 / cluster 8
     CUDA compiler:       /usr/local/cuda/bin/nvcc nvcc: NVIDIA (R) Cuda compiler driver;Copyright (c) 2005-2025 NVIDIA Corporation;Built on Fri_Feb_21_20:23:50_PST_2025;Cuda compilation tools, release 12.8, V12.8.93;Build cuda_12.8.r12.8/compiler.35583870_0
     CUDA compiler flags:-std=c++17;--generate-code=arch=compute_89,code=sm_89;-use_fast_math;-Xptxas;-warn-double-usage;-Xptxas;-Werror;-D_FORCE_INLINES;-Xcompiler;-fopenmp;-fexcess-precision=fast -funroll-all-loops -mavx512f -mfma -mavx512vl -mavx512dq -mavx512bw -Wno-missing-field-initializers -Wno-cast-function-type-strict SHELL:-fopenmp -O3 -DNDEBUG
     CUDA driver:         13.0
     CUDA runtime:        12.80

Test 3: GPU m

In [ ]:
# ==============================================================================
# BLOCK 3: SAFETY MONITOR MODE (Backs up every 30 mins) - MD REP 1
# ==============================================================================
import os
import shutil
import time
import glob
import threading

# --- CONFIGURATION ---
# We skip md_rep1 because it is already done
replicates = ["md_rep1"]
DRIVE_DIR = "/content/drive/MyDrive/96_MD"
# ---------------------

def background_backup(rep_name, interval=1800):
    """Copies files to Drive every 'interval' seconds (Default: 30 mins)."""
    while True:
        time.sleep(interval)
        print(f"\n[Auto-Backup] Syncing {rep_name} data to Drive...")

        # Find all files for this replicate (including parts)
        files = glob.glob(f"{rep_name}*")
        for file_path in files:
            # We copy files if they are newer/larger than the Drive version
            if os.path.isfile(file_path):
                filename = os.path.basename(file_path)
                drive_path = f"{DRIVE_DIR}/{filename}"

                # Simple Copy (rsync logic)
                try:
                    shutil.copy(file_path, drive_path)
                except Exception as e:
                    print(f"  Warning: Could not sync {filename} ({e})")

        print("[Auto-Backup] Done. Continuing simulation...\n")

        # Check if GROMACS is still running. If not, stop backing up.
        latest_log = get_latest_part(rep_name, ".log")
        if latest_log and (time.time() - os.path.getmtime(latest_log) > 600):
            break

def get_latest_part(rep_name, extension):
    """Finds the latest part file."""
    files = glob.glob(f"{rep_name}*{extension}")
    if not files: return None
    files.sort(key=os.path.getmtime)
    return files[-1]

# --- MAIN LOOP ---
for rep in replicates:
    print(f"\n{'='*60}")
    print(f"PROCESSING: {rep}")
    print(f"{'='*60}")

    # 1. CHECK IF DONE
    latest_gro = get_latest_part(rep, ".gro")
    if latest_gro and os.path.getsize(latest_gro) > 1000:
        print(f">>> ✅ {rep} finished. Skipping.")
        continue

    # 2. PREPARE INPUTS
    tpr_file = f"{rep}.tpr"
    if not os.path.exists(tpr_file):
        if os.path.exists(f"{DRIVE_DIR}/{tpr_file}"):
            shutil.copy(f"{DRIVE_DIR}/{tpr_file}", ".")
        else:
            print(f"!!! CRITICAL: {tpr_file} missing. Skipping.")
            continue

    # 3. SMART RESUME
    latest_cpt = get_latest_part(rep, ".cpt")
    resume_cmd = ""

    # Check Local first, then Drive (to recover from crash)
    if latest_cpt:
        print(f">>> Found local checkpoint: {latest_cpt}")
        resume_cmd = f"-cpi {latest_cpt} -noappend"
    else:
        # Check Drive for a rescue checkpoint
        drive_cpts = glob.glob(f"{DRIVE_DIR}/{rep}*.cpt")
        if drive_cpts:
            drive_cpts.sort(key=os.path.getmtime)
            last_drive_cpt = drive_cpts[-1]
            print(f">>> Found Drive checkpoint: {last_drive_cpt}")
            shutil.copy(last_drive_cpt, ".")
            resume_cmd = f"-cpi {os.path.basename(last_drive_cpt)} -noappend"
        else:
            print(">>> No checkpoint found. Starting FRESH.")

    # 4. START BACKUP THREAD (Every 30 Mins)
    stop_event = threading.Event()
    # Interval set to 1800 seconds (30 mins)
    backup_thread = threading.Thread(target=background_backup, args=(rep, 1800))
    backup_thread.daemon = True
    backup_thread.start()

    # 5. RUN GROMACS
    cmd = (
        f"gmx mdrun -deffnm {rep} "
        f"-nb gpu -pme gpu -pmefft gpu "
        f"-ntmpi 1 -ntomp {os.cpu_count()} "
        f"{resume_cmd}"
    )
    print(f">>> Executing: {cmd}")
    os.system(cmd)

    # 6. FINAL SYNC
    print(">>> Run finished. Performing final sync...")
    for file_path in glob.glob(f"{rep}*"):
        if os.path.isfile(file_path):
            shutil.copy(file_path, f"{DRIVE_DIR}/{os.path.basename(file_path)}")


PROCESSING: md_rep1
>>> No checkpoint found. Starting FRESH.
>>> Executing: gmx mdrun -deffnm md_rep1 -nb gpu -pme gpu -pmefft gpu -ntmpi 1 -ntomp 12 

[Auto-Backup] Syncing md_rep1 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep1 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep1 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep1 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep1 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep1 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep1 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep1 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep1 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[

In [ ]:
# ==============================================================================
# BLOCK: FORCE SAVE (Bypasses Drive Sync Lag) - SAVE REP 1
# ==============================================================================
import os
import shutil
import time

# --- CONFIGURATION ---
LOCAL_FILE = "/content/96_MD/md_rep1.xtc"
DRIVE_DIR = "/content/drive/MyDrive/96_MD"
NEW_NAME = "md_rep1_FINAL.xtc"  # New name forces a fresh upload
# ---------------------

if os.path.exists(LOCAL_FILE):
    local_size = os.path.getsize(LOCAL_FILE) / (1024*1024*1024) # In GB
    print(f">>> Local file found: {local_size:.2f} GB")

    dest_path = f"{DRIVE_DIR}/{NEW_NAME}"

    print(f">>> 🚀 Force-copying to: {NEW_NAME}")
    print(">>> This might take 2-5 minutes. Please wait...")

    # Force copy
    shutil.copy(LOCAL_FILE, dest_path)

    # Verify immediately
    if os.path.exists(dest_path):
        drive_size = os.path.getsize(dest_path) / (1024*1024*1024)
        print(f">>> ✅ Copy Complete.")
        print(f"    Local Size: {local_size:.2f} GB")
        print(f"    Drive Size: {drive_size:.2f} GB")

        if abs(local_size - drive_size) < 0.01:
            print(">>> SUCCESS: Files match perfectly.")
        else:
            print(">>> WARNING: Sizes still differ. Drive might still be processing.")
    else:
        print("!!! ERROR: Copy failed.")
else:
    print(f"!!! CRITICAL: Local file {LOCAL_FILE} not found!")

>>> Local file found: 2.53 GB
>>> 🚀 Force-copying to: md_rep1_FINAL.xtc
>>> This might take 2-5 minutes. Please wait...


In [ ]:
# ==============================================================================
# BLOCK 3: SAFETY MONITOR MODE (Backs up every 30 mins) - MD REP 2
# ==============================================================================
import os
import shutil
import time
import glob
import threading

# --- CONFIGURATION ---
# We skip md_rep1 because it is already done
replicates = ["md_rep2"]
DRIVE_DIR = "/content/drive/MyDrive/96_MD"
# ---------------------

def background_backup(rep_name, interval=1800):
    """Copies files to Drive every 'interval' seconds (Default: 30 mins)."""
    while True:
        time.sleep(interval)
        print(f"\n[Auto-Backup] Syncing {rep_name} data to Drive...")

        # Find all files for this replicate (including parts)
        files = glob.glob(f"{rep_name}*")
        for file_path in files:
            # We copy files if they are newer/larger than the Drive version
            if os.path.isfile(file_path):
                filename = os.path.basename(file_path)
                drive_path = f"{DRIVE_DIR}/{filename}"

                # Simple Copy (rsync logic)
                try:
                    shutil.copy(file_path, drive_path)
                except Exception as e:
                    print(f"  Warning: Could not sync {filename} ({e})")

        print("[Auto-Backup] Done. Continuing simulation...\n")

        # Check if GROMACS is still running. If not, stop backing up.
        latest_log = get_latest_part(rep_name, ".log")
        if latest_log and (time.time() - os.path.getmtime(latest_log) > 600):
            break

def get_latest_part(rep_name, extension):
    """Finds the latest part file."""
    files = glob.glob(f"{rep_name}*{extension}")
    if not files: return None
    files.sort(key=os.path.getmtime)
    return files[-1]

# --- MAIN LOOP ---
for rep in replicates:
    print(f"\n{'='*60}")
    print(f"PROCESSING: {rep}")
    print(f"{'='*60}")

    # 1. CHECK IF DONE
    latest_gro = get_latest_part(rep, ".gro")
    if latest_gro and os.path.getsize(latest_gro) > 1000:
        print(f">>> ✅ {rep} finished. Skipping.")
        continue

    # 2. PREPARE INPUTS
    tpr_file = f"{rep}.tpr"
    if not os.path.exists(tpr_file):
        if os.path.exists(f"{DRIVE_DIR}/{tpr_file}"):
            shutil.copy(f"{DRIVE_DIR}/{tpr_file}", ".")
        else:
            print(f"!!! CRITICAL: {tpr_file} missing. Skipping.")
            continue

    # 3. SMART RESUME
    latest_cpt = get_latest_part(rep, ".cpt")
    resume_cmd = ""

    # Check Local first, then Drive (to recover from crash)
    if latest_cpt:
        print(f">>> Found local checkpoint: {latest_cpt}")
        resume_cmd = f"-cpi {latest_cpt} -noappend"
    else:
        # Check Drive for a rescue checkpoint
        drive_cpts = glob.glob(f"{DRIVE_DIR}/{rep}*.cpt")
        if drive_cpts:
            drive_cpts.sort(key=os.path.getmtime)
            last_drive_cpt = drive_cpts[-1]
            print(f">>> Found Drive checkpoint: {last_drive_cpt}")
            shutil.copy(last_drive_cpt, ".")
            resume_cmd = f"-cpi {os.path.basename(last_drive_cpt)} -noappend"
        else:
            print(">>> No checkpoint found. Starting FRESH.")

    # 4. START BACKUP THREAD (Every 30 Mins)
    stop_event = threading.Event()
    # Interval set to 1800 seconds (30 mins)
    backup_thread = threading.Thread(target=background_backup, args=(rep, 1800))
    backup_thread.daemon = True
    backup_thread.start()

    # 5. RUN GROMACS
    cmd = (
        f"gmx mdrun -deffnm {rep} "
        f"-nb gpu -pme gpu -pmefft gpu "
        f"-ntmpi 1 -ntomp {os.cpu_count()} "
        f"{resume_cmd}"
    )
    print(f">>> Executing: {cmd}")
    os.system(cmd)

    # 6. FINAL SYNC
    print(">>> Run finished. Performing final sync...")
    for file_path in glob.glob(f"{rep}*"):
        if os.path.isfile(file_path):
            shutil.copy(file_path, f"{DRIVE_DIR}/{os.path.basename(file_path)}")


PROCESSING: md_rep2
>>> No checkpoint found. Starting FRESH.
>>> Executing: gmx mdrun -deffnm md_rep2 -nb gpu -pme gpu -pmefft gpu -ntmpi 1 -ntomp 12 

[Auto-Backup] Syncing md_rep2 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep2 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep2 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep2 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep2 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep2 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep2 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep2 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep2 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[

In [ ]:
# ==============================================================================
# BLOCK: FORCE SAVE (Bypasses Drive Sync Lag) - SAVE REP 2
# ==============================================================================
import os
import shutil
import time

# --- CONFIGURATION ---
LOCAL_FILE = "/content/96_MD/md_rep2.xtc"
DRIVE_DIR = "/content/drive/MyDrive/96_MD"
NEW_NAME = "md_rep2_FINAL.xtc"  # New name forces a fresh upload
# ---------------------

if os.path.exists(LOCAL_FILE):
    local_size = os.path.getsize(LOCAL_FILE) / (1024*1024*1024) # In GB
    print(f">>> Local file found: {local_size:.2f} GB")

    dest_path = f"{DRIVE_DIR}/{NEW_NAME}"

    print(f">>> 🚀 Force-copying to: {NEW_NAME}")
    print(">>> This might take 2-5 minutes. Please wait...")

    # Force copy
    shutil.copy(LOCAL_FILE, dest_path)

    # Verify immediately
    if os.path.exists(dest_path):
        drive_size = os.path.getsize(dest_path) / (1024*1024*1024)
        print(f">>> ✅ Copy Complete.")
        print(f"    Local Size: {local_size:.2f} GB")
        print(f"    Drive Size: {drive_size:.2f} GB")

        if abs(local_size - drive_size) < 0.01:
            print(">>> SUCCESS: Files match perfectly.")
        else:
            print(">>> WARNING: Sizes still differ. Drive might still be processing.")
    else:
        print("!!! ERROR: Copy failed.")
else:
    print(f"!!! CRITICAL: Local file {LOCAL_FILE} not found!")

>>> Local file found: 2.53 GB
>>> 🚀 Force-copying to: md_rep2_FINAL.xtc
>>> This might take 2-5 minutes. Please wait...
>>> ✅ Copy Complete.
    Local Size: 2.53 GB
    Drive Size: 2.53 GB
>>> SUCCESS: Files match perfectly.


In [ ]:
# ==============================================================================
# BLOCK 3: SAFETY MONITOR MODE (Backs up every 30 mins) - MD REP 3
# ==============================================================================
import os
import shutil
import time
import glob
import threading

# --- CONFIGURATION ---
# We skip md_rep1 because it is already done
replicates = ["md_rep3"]
DRIVE_DIR = "/content/drive/MyDrive/96_MD"
# ---------------------

def background_backup(rep_name, interval=1800):
    """Copies files to Drive every 'interval' seconds (Default: 30 mins)."""
    while True:
        time.sleep(interval)
        print(f"\n[Auto-Backup] Syncing {rep_name} data to Drive...")

        # Find all files for this replicate (including parts)
        files = glob.glob(f"{rep_name}*")
        for file_path in files:
            # We copy files if they are newer/larger than the Drive version
            if os.path.isfile(file_path):
                filename = os.path.basename(file_path)
                drive_path = f"{DRIVE_DIR}/{filename}"

                # Simple Copy (rsync logic)
                try:
                    shutil.copy(file_path, drive_path)
                except Exception as e:
                    print(f"  Warning: Could not sync {filename} ({e})")

        print("[Auto-Backup] Done. Continuing simulation...\n")

        # Check if GROMACS is still running. If not, stop backing up.
        latest_log = get_latest_part(rep_name, ".log")
        if latest_log and (time.time() - os.path.getmtime(latest_log) > 600):
            break

def get_latest_part(rep_name, extension):
    """Finds the latest part file."""
    files = glob.glob(f"{rep_name}*{extension}")
    if not files: return None
    files.sort(key=os.path.getmtime)
    return files[-1]

# --- MAIN LOOP ---
for rep in replicates:
    print(f"\n{'='*60}")
    print(f"PROCESSING: {rep}")
    print(f"{'='*60}")

    # 1. CHECK IF DONE
    latest_gro = get_latest_part(rep, ".gro")
    if latest_gro and os.path.getsize(latest_gro) > 1000:
        print(f">>> ✅ {rep} finished. Skipping.")
        continue

    # 2. PREPARE INPUTS
    tpr_file = f"{rep}.tpr"
    if not os.path.exists(tpr_file):
        if os.path.exists(f"{DRIVE_DIR}/{tpr_file}"):
            shutil.copy(f"{DRIVE_DIR}/{tpr_file}", ".")
        else:
            print(f"!!! CRITICAL: {tpr_file} missing. Skipping.")
            continue

    # 3. SMART RESUME
    latest_cpt = get_latest_part(rep, ".cpt")
    resume_cmd = ""

    # Check Local first, then Drive (to recover from crash)
    if latest_cpt:
        print(f">>> Found local checkpoint: {latest_cpt}")
        resume_cmd = f"-cpi {latest_cpt} -noappend"
    else:
        # Check Drive for a rescue checkpoint
        drive_cpts = glob.glob(f"{DRIVE_DIR}/{rep}*.cpt")
        if drive_cpts:
            drive_cpts.sort(key=os.path.getmtime)
            last_drive_cpt = drive_cpts[-1]
            print(f">>> Found Drive checkpoint: {last_drive_cpt}")
            shutil.copy(last_drive_cpt, ".")
            resume_cmd = f"-cpi {os.path.basename(last_drive_cpt)} -noappend"
        else:
            print(">>> No checkpoint found. Starting FRESH.")

    # 4. START BACKUP THREAD (Every 30 Mins)
    stop_event = threading.Event()
    # Interval set to 1800 seconds (30 mins)
    backup_thread = threading.Thread(target=background_backup, args=(rep, 1800))
    backup_thread.daemon = True
    backup_thread.start()

    # 5. RUN GROMACS
    cmd = (
        f"gmx mdrun -deffnm {rep} "
        f"-nb gpu -pme gpu -pmefft gpu "
        f"-ntmpi 1 -ntomp {os.cpu_count()} "
        f"{resume_cmd}"
    )
    print(f">>> Executing: {cmd}")
    os.system(cmd)

    # 6. FINAL SYNC
    print(">>> Run finished. Performing final sync...")
    for file_path in glob.glob(f"{rep}*"):
        if os.path.isfile(file_path):
            shutil.copy(file_path, f"{DRIVE_DIR}/{os.path.basename(file_path)}")


PROCESSING: md_rep3
>>> No checkpoint found. Starting FRESH.
>>> Executing: gmx mdrun -deffnm md_rep3 -nb gpu -pme gpu -pmefft gpu -ntmpi 1 -ntomp 12 

[Auto-Backup] Syncing md_rep3 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep3 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep3 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep3 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep3 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep3 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep3 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep3 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[Auto-Backup] Syncing md_rep3 data to Drive...
[Auto-Backup] Done. Continuing simulation...


[

In [ ]:
# ==============================================================================
# BLOCK: FORCE SAVE (Bypasses Drive Sync Lag) - SAVE REP 3
# ==============================================================================
import os
import shutil
import time

# --- CONFIGURATION ---
LOCAL_FILE = "/content/96_MD/md_rep3.xtc"
DRIVE_DIR = "/content/drive/MyDrive/96_MD"
NEW_NAME = "md_rep3_FINAL.xtc"  # New name forces a fresh upload
# ---------------------

if os.path.exists(LOCAL_FILE):
    local_size = os.path.getsize(LOCAL_FILE) / (1024*1024*1024) # In GB
    print(f">>> Local file found: {local_size:.2f} GB")

    dest_path = f"{DRIVE_DIR}/{NEW_NAME}"

    print(f">>> 🚀 Force-copying to: {NEW_NAME}")
    print(">>> This might take 2-5 minutes. Please wait...")

    # Force copy
    shutil.copy(LOCAL_FILE, dest_path)

    # Verify immediately
    if os.path.exists(dest_path):
        drive_size = os.path.getsize(dest_path) / (1024*1024*1024)
        print(f">>> ✅ Copy Complete.")
        print(f"    Local Size: {local_size:.2f} GB")
        print(f"    Drive Size: {drive_size:.2f} GB")

        if abs(local_size - drive_size) < 0.01:
            print(">>> SUCCESS: Files match perfectly.")
        else:
            print(">>> WARNING: Sizes still differ. Drive might still be processing.")
    else:
        print("!!! ERROR: Copy failed.")
else:
    print(f"!!! CRITICAL: Local file {LOCAL_FILE} not found!")

>>> Local file found: 2.53 GB
>>> 🚀 Force-copying to: md_rep3_FINAL.xtc
>>> This might take 2-5 minutes. Please wait...
>>> ✅ Copy Complete.
    Local Size: 2.53 GB
    Drive Size: 2.53 GB
>>> SUCCESS: Files match perfectly.


In [ ]:
# ==============================================================================
# BLOCK 3: BULLETPROOF PRODUCTION (Auto-Verify & Rescue)
# ==============================================================================
import os
import shutil
import time
import glob
import threading

# --- CONFIGURATION ---
replicates = ["md_rep3"]   # <--- ONLY REP 3
COMPOUND_FOLDER = "96_MD"  # <--- 96_MD
# ---------------------

DRIVE_DIR = f"/content/drive/MyDrive/{COMPOUND_FOLDER}"
LOCAL_DIR = f"/content/{COMPOUND_FOLDER}"

# Ensure we are in the right directory
if not os.path.exists(LOCAL_DIR): os.makedirs(LOCAL_DIR)
os.chdir(LOCAL_DIR)

# --- HELPER: ROBUST COPY FUNCTION ---
def robust_copy(src, dst):
    """Copies large files in chunks to avoid Drive timeouts."""
    try:
        with open(src, 'rb') as fsrc:
            with open(dst, 'wb') as fdst:
                while True:
                    buf = fsrc.read(10 * 1024 * 1024) # 10MB chunks
                    if not buf:
                        break
                    fdst.write(buf)
        return True
    except Exception as e:
        print(f"    ❌ Stream copy failed: {e}")
        return False

def verify_and_rescue(local_path, drive_path):
    """Checks file size match. If fail, saves as _RESCUED."""
    filename = os.path.basename(local_path)
    local_size = os.path.getsize(local_path)

    # 1. Check Primary Copy
    if os.path.exists(drive_path):
        drive_size = os.path.getsize(drive_path)
        if local_size == drive_size:
            print(f"    ✅ Verified: {filename} (Size matches)")
            return
        else:
            print(f"    ⚠️ MISMATCH: {filename} (Local: {local_size} vs Drive: {drive_size})")
    else:
        print(f"    ⚠️ MISSING: {filename} did not appear in Drive.")

    # 2. Rescue Strategy
    rescue_name = filename.replace(".xtc", "_RESCUED.xtc")
    if rescue_name == filename: rescue_name += "_RESCUED"

    rescue_path = f"{DRIVE_DIR}/{rescue_name}"
    print(f"    🚀 ATTEMPTING RESCUE SAVE: {rescue_name}...")

    success = robust_copy(local_path, rescue_path)

    if success and os.path.exists(rescue_path) and os.path.getsize(rescue_path) == local_size:
        print(f"    🎉 RESCUE SUCCESSFUL: {rescue_name} is safe.")
    else:
        print(f"    ❌ RESCUE FAILED. Please manual check {filename}.")

# --- BACKGROUND MONITOR ---
def background_backup(rep_name, interval=1800):
    while True:
        time.sleep(interval)
        print(f"\n[Auto-Backup] Syncing {rep_name}...")
        files = glob.glob(f"{rep_name}*")
        for file_path in files:
            # Skip massive files in background to prevent locking
            if file_path.endswith(".xtc") and os.path.getsize(file_path) > 1024*1024*1000:
                continue

            if os.path.isfile(file_path):
                try:
                    shutil.copy(file_path, f"{DRIVE_DIR}/{os.path.basename(file_path)}")
                except: pass

        # Stop if GROMACS is done (log not updated in 10 mins)
        latest_log = get_latest_part(rep_name, ".log")
        if latest_log and (time.time() - os.path.getmtime(latest_log) > 600):
            break

def get_latest_part(rep_name, extension):
    files = glob.glob(f"{rep_name}*{extension}")
    if not files: return None
    files.sort(key=os.path.getmtime)
    return files[-1]

# --- MAIN LOOP ---
for rep in replicates:
    print(f"\n{'='*60}")
    print(f"PROCESSING: {rep}")
    print(f"{'='*60}")

    # 1. INPUT CHECK
    tpr_file = f"{rep}.tpr"
    if not os.path.exists(tpr_file):
        if os.path.exists(f"{DRIVE_DIR}/{tpr_file}"):
            shutil.copy(f"{DRIVE_DIR}/{tpr_file}", ".")
        else:
            print(f"!!! CRITICAL: {tpr_file} missing. Skipping.")
            continue

    # 2. START FRESH (Since we nuked Rep 3)
    print(">>> Starting FRESH simulation...")

    # 3. START MONITOR
    stop_event = threading.Event()
    backup_thread = threading.Thread(target=background_backup, args=(rep, 1800))
    backup_thread.daemon = True
    backup_thread.start()

    # 4. RUN GROMACS
    cmd = (
        f"gmx mdrun -deffnm {rep} "
        f"-nb gpu -pme gpu -pmefft gpu "
        f"-ntmpi 1 -ntomp {os.cpu_count()} "
    )
    print(f">>> Executing: {cmd}")
    os.system(cmd)

    # 5. FINAL "PARANOID" SYNC
    print(f"\n{'='*60}")
    print(">>> Run finished. Starting PARANOID SYNC...")
    print(f"{'='*60}")

    for file_path in glob.glob(f"{rep}*"):
        if os.path.isfile(file_path):
            filename = os.path.basename(file_path)
            drive_path = f"{DRIVE_DIR}/{filename}"

            # Special handling for XTC (The Troublemakers)
            if filename.endswith(".xtc"):
                print(f"Syncing XTC: {filename}...")
                robust_copy(file_path, drive_path)
                verify_and_rescue(file_path, drive_path)
            else:
                # Normal copy for small files
                shutil.copy(file_path, drive_path)
                print(f"Saved: {filename}")

    print("\n>>> ✅ JOB COMPLETE.")


PROCESSING: md_rep3
>>> Starting FRESH simulation...
>>> Executing: gmx mdrun -deffnm md_rep3 -nb gpu -pme gpu -pmefft gpu -ntmpi 1 -ntomp 12 

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto-Backup] Syncing md_rep3...

[Auto

In [ ]:
!pip install -q MDAnalysis

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 6.3 MB/s eta 0:00:00


In [ ]:
# ==============================================================================
# BLOCK: UNIVERSAL PYTHON XTC AUDITOR (MDAnalysis)
# ==============================================================================
import MDAnalysis as mda
import os
import glob
import shutil
import warnings

# Suppress standard MDAnalysis warnings about missing topology
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
COMPOUND_FOLDER = "96_MD"  # <--- CHANGE TO "96_MD" FOR THE NEXT BATCH
# ══════════════════════════════════════════════════════════════════════════════

DRIVE_DIR = f"/content/drive/MyDrive/{COMPOUND_FOLDER}"
LOCAL_TEMP = "/content/temp_audit.xtc"

print(f"📊 STARTING PYTHON AUDIT FOR: {COMPOUND_FOLDER}")
print(f"   Source: {DRIVE_DIR}")
print("="*85)
print(f"{'FILENAME':<40} | {'SIZE (GB)':<10} | {'FRAMES':<10} | {'DURATION':<10} | {'STATUS'}")
print("-" * 85)

# 1. Get all XTC files
files = sorted(glob.glob(f"{DRIVE_DIR}/*.xtc"))

if not files:
    print(f"❌ No .xtc files found in {DRIVE_DIR}")
else:
    for file_path in files:
        filename = os.path.basename(file_path)
        file_size_gb = os.path.getsize(file_path) / (1024**3)

        # 2. Copy to Local (Force Download)
        try:
            if os.path.exists(LOCAL_TEMP): os.remove(LOCAL_TEMP)
            shutil.copy(file_path, LOCAL_TEMP)
        except Exception as e:
            print(f"{filename:<40} | {file_size_gb:.2f}       | {'-':<10} | {'-':<10} | ❌ COPY FAIL")
            continue

        # 3. Analyze with Python
        try:
            # Load Universe (XTC only)
            u = mda.Universe(LOCAL_TEMP)

            n_frames = len(u.trajectory)
            dt = u.trajectory.dt  # Time step in ps

            # Calculate total time in ns
            total_ns = (n_frames * dt) / 1000

            # Status Logic
            status = "✅ OK"
            if total_ns > 99.0:
                status = "🎉 COMPLETE"
            elif total_ns < 1.0:
                status = "⚠️ TINY"

            print(f"{filename:<40} | {file_size_gb:.2f}       | {n_frames:<10} | {total_ns:.2f} ns    | {status}")

        except Exception as e:
             print(f"{filename:<40} | {file_size_gb:.2f}       | {'-':<10} | {'-':<10} | ❌ CORRUPT")

        # 4. Cleanup
        if os.path.exists(LOCAL_TEMP): os.remove(LOCAL_TEMP)

print("-" * 85)
print("Audit Complete.")

📊 STARTING PYTHON AUDIT FOR: 96_MD
   Source: /content/drive/MyDrive/96_MD
FILENAME                                 | SIZE (GB)  | FRAMES     | DURATION   | STATUS
-------------------------------------------------------------------------------------
md_rep1.xtc                              | 1.95       | 7721       | 77.21 ns    | ✅ OK
md_rep1_FINAL.xtc                        | 2.53       | 10001      | 100.01 ns    | 🎉 COMPLETE
md_rep2.part0002.xtc                     | 0.81       | 3191       | 31.91 ns    | ✅ OK
md_rep2.part0003.xtc                     | 1.15       | 4542       | 45.42 ns    | ✅ OK
md_rep2.xtc                              | 0.27       | 1082       | 10.82 ns    | ✅ OK
md_rep2_part-0003FINAL.xtc               | 1.44       | 5711       | 57.11 ns    | ✅ OK
md_rep3.xtc                              | 0.94       | 3729       | 37.29 ns    | ✅ OK
md_rep3_FINAL.xtc                        | 2.53       | 10001      | 100.01 ns    | 🎉 COMPLETE
--------------------------------